In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.ndimage import gaussian_filter
from scipy.stats import ttest_ind, mannwhitneyu

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
rev_class = list(np.loadtxt(f'{projdir}/txt/clusters/rev_class.txt', dtype=str))

In [ ]:
pmds_path = f'{projdir}/pmds/dnmtools/rna_cluster_donor/merged/start_merged.filt_mcg.sorted.bed.gz'
pmds = pd.read_csv(pmds_path, sep='\t', header=None)
pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
pmds.shape

In [ ]:
pmds.head()

In [ ]:
def process_bed(s):
    pmds = pd.read_csv(f'{projdir}/bed/pmds/rev_class/{s}.start_pmds_CGN-Both.full.bed.gz', sep='\t', header=None)
    pmds.index = [f'{x}_{y}' for x,y in zip(pmds[0], pmds[1])]
    return pmds

with ProcessPoolExecutor() as executor:
    pmds_results = list(tqdm(executor.map(process_bed, rev_class), total=len(rev_class)))

In [ ]:
pmds_results[0].head()

In [ ]:
mc_results = [x[3] / x[4] for x in pmds_results]
pmds_df = pd.concat(mc_results, axis=1)
pmds_df.columns = rev_class

for s in rev_class:
    pmds_df[s].fillna(pmds_df[s].mean())
    
pmds_df.shape

In [ ]:
pmds_df.head()

In [ ]:
whitelist = ['Start', 'Inter2', 'Fail2', 'IPS']
mask = [x.split('_')[0] in whitelist for x in pmds_df.columns]
final_df = pmds_df.loc[:,mask]

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12,5))
ax = sns.heatmap(final_df, vmin=0.2, vmax=0.7, cmap='coolwarm')
ax.set_yticklabels([]);

In [ ]:
ax = sns.clustermap(final_df, vmin=0.2, vmax=0.7, cmap='coolwarm')

### specific clusters

In [ ]:
whitelist = ['Start_1', 'Start_2', 'Fail2_3', 'Inter2_3', 'IPS_2', 'IPS_3']
mask = [x in whitelist for x in pmds_df.columns]
final_df = pmds_df.loc[:,mask]
final_df = final_df[whitelist]

In [ ]:
ax = sns.clustermap(final_df, vmin=0.2, vmax=0.7, cmap='coolwarm', col_cluster=False)

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
final_df.to_csv(f'{outdir}/rev_class_pmd_mcg.csv', sep='\t')

In [ ]:
!date